# Boeing BCAI - Memory Store
### VS Code / local Jupyter edition - `BoeingChatModel` (no direct OpenAI calls)

Same conversation-memory patterns as the original notebook, rebuilt on `BoeingChatModel`
instead of `openai.OpenAI`:

1. Basic call, no memory (each call is independent)
2. Full conversation history sent every turn
3. Interactive chat with full history (`input()` loop)
4. Conversation summary memory (running summary instead of full history)
5. Last-N-turns memory (sliding window)

Every LLM call goes through `BoeingChatModel`. No `import openai` anywhere in this notebook.

**Local setup differences from Colab:**
- Token comes from a `.env` file (via `python-dotenv`), not Colab Secrets.
- `boeing_chat_model.py` / `boeing_embeddings.py` are expected to sit next to this notebook, not uploaded at runtime.
- Dependencies are installed once into your local/virtual environment ahead of time, not via a `!pip install` cell.
- The `input()` loops below run in VS Code's interactive Jupyter cell input box the same way they run in Colab.

**Before running:** create a `.env` file in the same folder as this notebook:

```
UDAL_PAT=your_actual_token_here
```

Then select that environment's kernel in VS Code (Command Palette -> "Jupyter: Select Interpreter to Start Jupyter Server").


## 0. Setup

Install once in your terminal (not as a notebook cell):

```bash
pip install langchain-core pydantic httpx requests python-dotenv
```


In [ ]:
import os
from pathlib import Path

required = ["boeing_chat_model.py", "boeing_embeddings.py"]
missing = [f for f in required if not Path(f).exists()]
assert not missing, (
    f"Missing: {missing}. Place these files in the same folder as this notebook before continuing."
)
print("Wrapper files present:", required)


In [ ]:
from dotenv import load_dotenv

# Loads variables from a .env file in the same folder as this notebook into the environment.
load_dotenv()

UDAL_PAT = os.getenv("UDAL_PAT")
assert UDAL_PAT, "Set UDAL_PAT in a .env file in this folder before continuing."
print("UDAL_PAT loaded:", UDAL_PAT[:4] + "..." + UDAL_PAT[-4:])


In [ ]:
from boeing_chat_model import BoeingChatModel
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = BoeingChatModel(udal_pat=UDAL_PAT, model="gpt-4.1-mini", temperature=0.7, max_tokens=250)

print("BoeingChatModel ready:", llm._llm_type, "| model =", llm.model)


## 1. Basic call, no memory

Original notebook used `client.chat.completions.create(messages=[{"role": "user", ...}])`
with a single message. `BoeingChatModel.invoke()` takes a list of LangChain message objects
instead of raw dicts -- same idea, typed messages.


In [ ]:
response = llm.invoke([
    HumanMessage(content="I Am Ahmed, Here in Chennai")
])

print(response.content)


### 1.1 Second call with no memory carried over

Same as the original -- a brand-new call with no reference to the previous turn. Since nothing
was stored, the model has no way to know the answer.


In [ ]:
response = llm.invoke([
    HumanMessage(content="Whats my name")
])

print(response.content)


### 1.2 `max_tokens` cutoff example

Original notebook capped `max_tokens=10` to show a truncated response. Same effect here via a
one-off `BoeingChatModel` instance with a low `max_tokens`.


In [ ]:
short_llm = BoeingChatModel(udal_pat=UDAL_PAT, model="gpt-4.1-mini", max_tokens=10)

response = short_llm.invoke([
    HumanMessage(content="What's Football in 1000 words")
])

print(response.content)


## 2. Conversation container

Original notebook used a plain dict `{"messages": [...]}` of role/content dicts. Here the same
container holds LangChain message *objects* instead of dicts, since that's what
`BoeingChatModel.invoke()` expects.


In [ ]:
conversation = {
    "messages": []
}


## 3. Full history memory -- interactive loop

Every turn appends the new user message, sends the **entire** accumulated message list to
`BoeingChatModel`, then appends the assistant's reply. This is the simplest memory strategy --
correct, but the messages list (and token cost) grows every turn.

Type `exit` or `quit` to end the loop, same as the original. In VS Code, the `input()` prompt
appears in a small box at the top of the editor when this cell runs.


In [ ]:
conversation = {"messages": []}

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    # Add user's message to conversation
    conversation["messages"].append(HumanMessage(content=user_input))

    # Send entire conversation to BoeingChatModel
    response = llm.invoke(conversation["messages"])

    # Extract AI response
    assistant_response = response.content

    # Add AI response to conversation
    conversation["messages"].append(AIMessage(content=assistant_response))

    print("AI:", assistant_response)
    print("-" * 50)

print("\nConversation History:")
print(conversation)


## 4. Conversation summary memory

Instead of storing the full transcript, keep a single running text summary. Each turn:
1. Build a prompt from the summary + the new question.
2. Get the answer from `llm`.
3. Ask a second `BoeingChatModel` call to fold the latest exchange into an updated summary.

Same two-call-per-turn structure as the original (one for the answer, one to update the summary).


In [ ]:
conversation_summary = ""

while True:

    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    # Create the prompt using only the summary + latest question
    messages = [
        SystemMessage(content=(
            "You are a helpful conversational AI. "
            "Use the conversation summary to maintain context. "
            "Answer the user's latest question clearly and concisely."
        )),
        HumanMessage(content=(
            f"Previous Conversation Summary:\n{conversation_summary}\n\n"
            f"New User Question:\n{user_input}"
        )),
    ]

    response = llm.invoke(messages)
    assistant_response = response.content

    print("\nAI:", assistant_response)

    # Display token usage
    usage = response.response_metadata.get("usage", {})

    print("\n--- Token Usage ---")
    print("Input Tokens:", usage.get("prompt_tokens"))
    print("Output Tokens:", usage.get("completion_tokens"))
    print("Total Tokens:", usage.get("total_tokens"))

    # Update the conversation summary
    summary_response = llm.invoke([
        SystemMessage(content=(
            "Summarize the conversation in simple, crisp language. "
            "Keep only information that is useful for maintaining "
            "future conversational context. "
            "Do not include unnecessary details."
        )),
        HumanMessage(content=(
            f"Existing Summary:\n{conversation_summary}\n\n"
            f"New User Question:\n{user_input}\n\n"
            f"AI Response:\n{assistant_response}"
        )),
    ])

    conversation_summary = summary_response.content

    print("\n--- Current Summary ---")
    print(conversation_summary)
    print("=" * 60)


## 5. Last-N-turns memory (sliding window)

Keep only the most recent `MAX_TURNS` user/assistant exchanges. After each response, trim the
message list down to `MAX_TURNS * 2` messages (one user + one assistant per turn).


In [ ]:
conversation = []

MAX_TURNS = 3

while True:

    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    # Add user message
    conversation.append(HumanMessage(content=user_input))

    response = llm.invoke(conversation)
    assistant_response = response.content

    # Add AI response
    conversation.append(AIMessage(content=assistant_response))

    print("\nAI:", assistant_response)

    # Token information
    usage = response.response_metadata.get("usage", {})
    print("\n--- Token Usage ---")
    print("Input Tokens:", usage.get("prompt_tokens"))
    print("Output Tokens:", usage.get("completion_tokens"))
    print("Total Tokens:", usage.get("total_tokens"))

    # Keep only the latest N turns
    conversation = conversation[-(MAX_TURNS * 2):]

    print("\n--- Stored Messages ---")
    print(len(conversation))

    print("=" * 60)


## 6. Wrap-up

Same three memory strategies as the original notebook -- full history, running summary,
sliding window -- rebuilt on `BoeingChatModel` and LangChain message objects instead of the
raw OpenAI SDK and role/content dicts. Token usage is read the same way in all cases via
`response.response_metadata["usage"]`.

**Checkpoint - you should be able to:**
- Explain why full-history memory is simplest but grows unbounded
- Explain how conversation-summary memory keeps token usage roughly flat across a long conversation
- Explain how sliding-window (last-N-turns) memory differs from summary memory (loses old detail entirely vs. compresses it)
